# Panel 1B: Extracción de Características, EDA de Negocio y Segmentación K-Means

Este cuaderno constituye la **Fase 2 del Panel 1** y parte **exclusivamente de los datos limpios y auditados (`datasets/limpio/`)** generados en el Cuaderno 1A.

### Componentes del Cuaderno:
1. **Análisis Exploratorio de Negocio (Macro-EDA):** Comportamiento de ventas por día, preferencia de métodos de pago (**66.3% Efectivo vs 33.7% Yape**) y salud del inventario con alertas de quiebre reales.
2. **Extracción de Características por Ticket (`Feature Engineering`):**
   - Al haberse determinado que la hora de compra no es representativa (por registro en lote los fines de semana), se construye una matriz de características de **comportamiento real del comprador**:
     * `Total` (Monto pagado S/).
     * `n_items` (Volumen físico de unidades compradas).
     * `diversidad_productos` (Cantidad de productos distintos en el ticket).
3. **Detección Estadística de Outliers (1.5·IQR):** Identificación visual y analítica de compras mayoristas/institucionales.
4. **Segmentación K-Means:** Evaluación óptima con **Método del Codo** y **Coeficiente de Silueta** ($K=3$) y perfilamiento de clientes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print("Librerías de análisis y clustering importadas.")

## 1. Carga de Datos Limpios (`datasets/limpio/`)
Verificamos la carga de la data saneada con cero valores nulos y tipos normalizados.

In [ ]:
limpio_dir = 'datasets/limpio'
df_ventas = pd.read_csv(f'{limpio_dir}/ventas.csv')
df_detalle = pd.read_csv(f'{limpio_dir}/detalle_ventas.csv')
df_inv = pd.read_csv(f'{limpio_dir}/inventario.csv')

df_ventas['Fecha'] = pd.to_datetime(df_ventas['Fecha'])
print(f"[OK] Ventas limpias: {df_ventas.shape}")
print(f"[OK] Detalle limpio: {df_detalle.shape}")
print(f"[OK] Inventario limpio: {df_inv.shape}")
print("Primeras filas de ventas limpias:")
print(df_ventas.head(3))

## 2. Análisis Exploratorio de Negocio (Macro-EDA)
### 2.1 KPIs Operativos y Preferencia de Pago

In [ ]:
total_ingresos = df_ventas['Total'].sum()
ticket_promedio = df_ventas['Total'].mean()
n_transacciones = len(df_ventas)

print(f"--- RESUMEN FINANCIERO DEL BAZAR ---")
print(f"Ingresos Totales Acumulados: S/ {total_ingresos:,.2f}")
print(f"Total de Tickets Emitidos:    {n_transacciones:,}")
print(f"Ticket Promedio (Media):      S/ {ticket_promedio:,.2f}")

pago_dist = df_ventas['Metodo_Pago'].value_counts()
pago_pct = df_ventas['Metodo_Pago'].value_counts(normalize=True) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=pago_dist.index, y=pago_dist.values, hue=pago_dist.index, ax=ax1, palette=['#2b5c8f', '#d95f02'], legend=False)
ax1.set_title('Volumen por Método de Pago')
ax1.set_ylabel('Transacciones')

ax2.pie(pago_pct, labels=pago_pct.index, autopct='%1.1f%%', colors=['#2b5c8f', '#d95f02'], startangle=140)
ax2.set_title('Proporción de Uso de Método de Pago')
plt.tight_layout()
plt.show()

### 2.2 Demanda por Día de la Semana
Al haberse descartado la hora por la digitación de fin de semana, analizamos el comportamiento diario de las ventas registradas.

In [ ]:
df_ventas['Dia_Semana'] = df_ventas['Fecha'].dt.day_name()
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 4))
sns.countplot(data=df_ventas, x='Dia_Semana', order=[d for d in orden_dias if d in df_ventas['Dia_Semana'].unique()], color='#2ecc71')
plt.title('Distribución de Tickets por Día de la Semana')
plt.xlabel('Día de la Semana')
plt.ylabel('Cantidad de Ventas')
plt.show()

### 2.3 Valorización de Almacén y Alertas Reales de Reposición
Sobre el inventario saneado identificamos los artículos con `Stock_Actual <= Stock_Minimo` que requieren orden de compra urgente.

In [ ]:
df_inv['Valor_Inventario'] = df_inv['Stock_Actual'] * df_inv['Costo_Unitario']
valor_almacen = df_inv['Valor_Inventario'].sum()
alertas = df_inv[df_inv['Stock_Actual'] <= df_inv['Stock_Minimo']]

print(f"Valor Total en Almacén (Saneado): S/ {valor_almacen:,.2f}")
print(f"Artículos con Alerta de Reposición Urgente: {len(alertas)} de {len(df_inv)} ({len(alertas)/len(df_inv):.1%})")
print("\nTop 5 Ítems con Stock Crítico:")
print(alertas[['Descripcion', 'Departamento', 'Stock_Minimo', 'Stock_Actual']].head(5).to_string(index=False))

## 3. Extracción de Características por Ticket (`Feature Engineering`)
En reemplazo de la variable horaria (ruido artificial), creamos una matriz representativa del **comportamiento real de compra** de cada ticket:
- `Total`: Monto en Soles.
- `n_items`: Unidades físicas totales adquiridas.
- `diversidad_productos`: Cantidad de productos únicos dentro de la misma compra.

In [ ]:
agg_ticket = df_detalle.groupby('ID_Venta').agg(
    n_items=('Cantidad', 'sum'),
    diversidad_productos=('ID_Producto', 'nunique')
).reset_index()

df_cluster = df_ventas.merge(agg_ticket, left_on='ID', right_on='ID_Venta', how='inner')
features = ['Total', 'n_items', 'diversidad_productos']
X = df_cluster[features].copy()

print("Primeras filas de las Características Extraídas por Ticket:")
print(X.head())
print("\nEstadísticas descriptivas de las Características:")
print(X.describe().round(2))

## 4. Detección Estadística de Outliers (Regla 1.5·IQR)
Calculamos los cuartiles Q1, Q3 y el umbral superior para aislar compras institucionales o mayoristas.

In [ ]:
q1 = X['Total'].quantile(0.25)
q3 = X['Total'].quantile(0.75)
iqr = q3 - q1
umbral_superior = q3 + 1.5 * iqr

outliers = df_cluster[df_cluster['Total'] > umbral_superior]
print(f"Q1: S/ {q1:.2f} | Q3: S/ {q3:.2f} | IQR: S/ {iqr:.2f}")
print(f"Umbral de Outlier Superior: S/ {umbral_superior:.2f}")
print(f"Tickets Outliers Detectados: {len(outliers)} ({len(outliers)/len(df_cluster):.2%})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=df_cluster['Total'], ax=ax1, color='#e74c3c')
ax1.set_title('Boxplot de Ventas con Umbral Outlier')
ax1.axhline(umbral_superior, color='black', linestyle='--', label=f'Umbral S/ {umbral_superior:.1f}')
ax1.legend()

sns.histplot(df_cluster['Total'], bins=40, kde=True, ax=ax2, color='#34495e')
ax2.set_title('Distribución de Montos con Umbral')
ax2.axvline(umbral_superior, color='red', linestyle='--', label='Umbral Outlier')
ax2.legend()
plt.tight_layout()
plt.show()

## 5. Segmentación de Clientes con K-Means
### 5.1 Estandarización y Selección de K (Codo y Silueta)
Estandarizamos las variables con `StandardScaler` y justificamos la elección de $K=3$ analizando la inercia y el score de silueta.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
silhouettes = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
ax1.plot(k_range, inertias, 'bo-', marker='o', linewidth=2)
ax1.set_title('Método del Codo (Inercia vs K)')
ax1.set_xlabel('Número de Clústeres (K)')
ax1.set_ylabel('Inercia')
ax1.grid(True)

ax2.plot(k_range, silhouettes, 'ro-', marker='s', linewidth=2)
ax2.set_title('Coeficiente de Silueta vs K')
ax2.set_xlabel('Número de Clústeres (K)')
ax2.set_ylabel('Score de Silueta')
ax2.grid(True)
plt.tight_layout()
plt.show()

### 5.2 Ajuste Final ($K=3$) y Perfilamiento Ejecutivo del Negocio
Entrenamos el modelo con 3 clústeres e interpretamos cada segmento para la estrategia comercial del bazar.

In [ ]:
K_OPTIMO = 3
kmeans = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)

print(f"Coeficiente de Silueta Final para K={K_OPTIMO}: {silhouette_score(X_scaled, df_cluster['Cluster']):.4f}")

plt.figure(figsize=(9, 5))
sns.scatterplot(data=df_cluster, x='Total', y='n_items', hue='Cluster', palette='Set1', style='Cluster', s=80, alpha=0.85)
plt.title(f'Segmentación K-Means de Comportamiento de Compra (K={K_OPTIMO})')
plt.xlabel('Monto Total del Ticket (S/)')
plt.ylabel('Cantidad Total de Ítems Adquiridos')
plt.show()

In [ ]:
perfiles = df_cluster.groupby('Cluster')[['Total', 'n_items', 'diversidad_productos']].mean().round(2)
perfiles['Cantidad_Tickets'] = df_cluster['Cluster'].value_counts()
perfiles['Participacion_%'] = (perfiles['Cantidad_Tickets'] / len(df_cluster) * 100).round(1)
print("=== PERFILAMIENTO EJECUTIVO DE LOS CLÚSTERES DE CLIENTES ===")
print(perfiles)